In [1]:
import os
import numpy as np
import cupy as cp
from scipy.signal import find_peaks, sosfilt, butter, sosfilt_zi
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

PROCESSED_DIR = r"C:\Users\RF-LAB\Desktop\Pipeline\Train_iq_preprocessed"
OUTPUT_CSV    = r"C:\Users\RF-LAB\Desktop\Pipeline\Features_DroneRF.csv"

fs            = 100e6
BURST_THRESH  = 0.3
NPERSEG       = 1024
HOP           = 512
MAX_WORKERS   = 1       # Single thread — GPU memory is the bottleneck

In [3]:
# ============================================================
# VRAM CONTROL — most important setting
# ============================================================
# Each file = 100M IQ pairs. We subsample to fit in VRAM.
# 5M complex float32 samples = ~40MB VRAM — safe for RTX 3050
# Features computed on subsample are statistically stable.
# Increase to 10_000_000 only if no OOM errors.

MAX_SAMPLES = 5_000_000

In [4]:
# ============================================================
# Feature functions — GPU where beneficial, CPU otherwise
# ============================================================

'''
Spectral Features (GPU)

gpu_spectral_flatness()
gpu_spectral_entropy()

Both operate on the power spectrum computed as:
FFT → power spectrum → flatness/entropy'''

def gpu_spectral_flatness(power_spec_gpu):
    ps  = cp.abs(power_spec_gpu) + 1e-12
    geo = cp.exp(cp.mean(cp.log(ps)))
    ari = cp.mean(ps)
    return float(geo / ari)


def gpu_spectral_entropy(power_spec_gpu):
    ps      = cp.abs(power_spec_gpu) + 1e-12
    ps_norm = ps / cp.sum(ps)
    return float(-cp.sum(ps_norm * cp.log2(ps_norm)))

'''
Autocorrelation peak:
Purpose:

Detects periodicity in the IQ signal.
Useful for repetitive radar returns.'''

def cpu_correlation_peak(z_cpu):
    """
    Computed on CPU — FFT of 200M samples crashes 4GB VRAM.
    Uses shorter subsample — correlation peak is stable on 1M samples.
    """
    # Use max 1M samples for autocorrelation — more than sufficient
    seg   = z_cpu[:1_000_000]
    N     = len(seg)
    Z     = np.fft.fft(seg, n=2*N)
    acorr = np.fft.ifft(Z * np.conj(Z)).real[:N]
    acorr /= (acorr[0] + 1e-12)
    peaks, _ = find_peaks(acorr[1:], height=0.05)
    if len(peaks) > 0:
        return float(acorr[peaks[0] + 1])
    return 0.0

'''
Power Variance:
Measures how average power fluctuates across windows → indicates modulation or activity bursts.'''

def gpu_power_variance(z_gpu, window_size=1024):
    
    power    = cp.abs(z_gpu) ** 2
    n_blocks = len(power) // window_size
    if n_blocks == 0:
        return float(cp.var(power))
    blocks      = power[:n_blocks * window_size].reshape(n_blocks, window_size)
    block_means = cp.mean(blocks, axis=1)
    return float(cp.var(block_means))

'''
Burst Features
Computes:

number of bursts
mean burst duration (ms)
std deviation of duration
duty cycle'''

def gpu_burst_features(z_gpu, fs, threshold=BURST_THRESH):
    amplitude  = cp.abs(z_gpu)
    max_amp    = cp.max(amplitude)
    active     = (amplitude > (threshold * max_amp)).astype(cp.int32)
    active_cpu = cp.asnumpy(active)
    diff       = np.diff(active_cpu)

    burst_starts = np.where(diff ==  1)[0]
    burst_ends   = np.where(diff == -1)[0]

    duty = float(cp.mean(active.astype(cp.float32)))

    if len(burst_starts) == 0 or len(burst_ends) == 0:
        return 0, 0.0, 0.0, duty

    if burst_ends[0] < burst_starts[0]:
        burst_ends = burst_ends[1:]
    if len(burst_starts) > len(burst_ends):
        burst_starts = burst_starts[:len(burst_ends)]
    if len(burst_starts) == 0:
        return 0, 0.0, 0.0, duty

    durations_ms = (burst_ends - burst_starts) / fs * 1000.0
    return (int(len(durations_ms)),
            float(np.mean(durations_ms)),
            float(np.std(durations_ms)),
            duty)

'''
gpu_micro_doppler()
Uses a short-time FFT (STFT) to compute:

mean micro-Doppler energy
std deviation
blade rate (propeller modulation frequency)
'''

def gpu_micro_doppler(z_gpu, fs, nperseg=NPERSEG, hop=HOP):
    """
    Uses only first 500K samples for STFT — sufficient for blade rate.
    Avoids building a huge frame matrix on GPU.
    """
    seg     = z_gpu[:500_000]
    window  = cp.hanning(nperseg).astype(cp.float32)
    n_steps = (len(seg) - nperseg) // hop + 1

    if n_steps <= 0:
        return 0.0, 0.0, 0.0

    # Process in batches to avoid VRAM spike
    batch_size   = 500
    energy_list  = []

    for batch_start in range(0, n_steps, batch_size):
        batch_end = min(batch_start + batch_size, n_steps)
        frames = cp.stack([
            seg[i*hop : i*hop + nperseg] * window
            for i in range(batch_start, batch_end)
        ])                                          # (batch, nperseg)
        Zxx    = cp.fft.fft(frames, axis=1)
        energy = cp.sum(cp.abs(Zxx), axis=1)        # (batch,)
        energy_list.append(cp.asnumpy(energy))
        del frames, Zxx, energy
        cp.get_default_memory_pool().free_all_blocks()

    energy_cpu   = np.concatenate(energy_list)
    md_mean      = float(np.mean(energy_cpu))
    md_std       = float(np.std(energy_cpu))

    # Blade rate from energy envelope FFT
    energy_fft   = np.abs(np.fft.rfft(energy_cpu))
    energy_freqs = np.fft.rfftfreq(len(energy_cpu), d=hop/fs)
    mask         = (energy_freqs >= 10) & (energy_freqs <= 300)
    if np.any(mask):
        blade_rate = float(energy_freqs[mask][np.argmax(energy_fft[mask])])
    else:
        blade_rate = 0.0

    return md_mean, md_std, blade_rate


'''
gpu_doppler_shift()
Instantaneous frequency = derivative of phase.
Measures:

mean Doppler shift
variance in Doppler

Useful for speed estimation.'''

def gpu_doppler_shift(z_gpu, fs):
    phase     = cp.unwrap(cp.angle(z_gpu))
    inst_freq = cp.diff(phase) / (2.0 * cp.pi) * fs
    return float(cp.mean(inst_freq)), float(cp.std(inst_freq))


'''
gpu_range_fft()
Computes batched FFTs over windows of the signal to extract:

peak range bin
power of that bin'''

def gpu_range_fft(z_gpu, nperseg=NPERSEG):
    """
    Batched range FFT — processes segments in small batches.
    Avoids allocating one huge matrix on GPU.
    """
    n_segments  = len(z_gpu) // nperseg
    if n_segments == 0:
        return 0.0, 0.0

    window      = cp.hanning(nperseg).astype(cp.float32)
    batch_size  = 1000
    avg_profile = cp.zeros(nperseg, dtype=cp.float32)
    n_batches   = 0

    for batch_start in range(0, n_segments, batch_size):
        batch_end   = min(batch_start + batch_size, n_segments)
        seg_matrix  = cp.stack([
            z_gpu[i*nperseg:(i+1)*nperseg]
            for i in range(batch_start, batch_end)
        ])                                              # (batch, nperseg)
        windowed    = seg_matrix * window
        range_batch = cp.abs(cp.fft.fft(windowed, axis=1))
        avg_profile += cp.mean(range_batch, axis=0)
        n_batches   += 1
        del seg_matrix, windowed, range_batch
        cp.get_default_memory_pool().free_all_blocks()

    avg_profile /= n_batches
    peak_bin     = int(cp.argmax(avg_profile))
    peak_power   = float(avg_profile[peak_bin])

    del avg_profile
    cp.get_default_memory_pool().free_all_blocks()

    return float(peak_bin), peak_power


In [5]:
# ============================================================
# Per-file feature extractor
# ============================================================

def extract_features(filepath):
    """
    Loads .npy file, subsamples to MAX_SAMPLES,
    computes all features with controlled VRAM usage.
    """
    data = np.load(filepath) 
    # Handle both formats:
    # Drone files:   shape (2, N) → row 0 = I, row 1 = Q
    # No-drone files: shape (N,) complex → extract real/imag

    if data.ndim == 2:
        I = data[0].astype(np.float32)
        Q = data[1].astype(np.float32)
    elif np.iscomplexobj(data):
        I = data.real.astype(np.float32)
        Q = data.imag.astype(np.float32)
    else:
        raise ValueError(f"Unexpected .npy shape: {data.shape}, dtype: {data.dtype}")
    

    # Subsample — take evenly spaced samples across the full signal
    # This preserves statistical properties better than just taking
    # the first N samples (avoids startup silence bias)
    N = len(I)
    if N > MAX_SAMPLES:
        idx = np.linspace(0, N-1, MAX_SAMPLES, dtype=np.int64)
        I   = I[idx]
        Q   = Q[idx]

    # CPU arrays kept for CPU-side features
    z_cpu = I + 1j * Q

    # Send subsampled signal to GPU
    z_gpu = cp.asarray(I) + 1j * cp.asarray(Q)
    del I, Q

    # ── Spectral features — on GPU subsample ──
    window_gpu = cp.hanning(len(z_gpu)).astype(cp.float32)
    Z_gpu      = cp.fft.fft(z_gpu * window_gpu)
    power_spec = cp.abs(Z_gpu) ** 2
    del window_gpu, Z_gpu
    cp.get_default_memory_pool().free_all_blocks()

    sf  = gpu_spectral_flatness(power_spec)
    se  = gpu_spectral_entropy(power_spec)
    del power_spec
    cp.get_default_memory_pool().free_all_blocks()

    # ── Correlation peak — CPU (more VRAM efficient) ──
    cp_ = cpu_correlation_peak(z_cpu)

    # ── Remaining GPU features ──
    pv                      = gpu_power_variance(z_gpu)
    bc, bd_mean, bd_std, dc = gpu_burst_features(z_gpu, fs)
    md_mean, md_std, blade  = gpu_micro_doppler(z_gpu, fs)
    dop_mean, dop_std       = gpu_doppler_shift(z_gpu, fs)
    r_bin, r_pwr            = gpu_range_fft(z_gpu)

    del z_gpu
    cp.get_default_memory_pool().free_all_blocks()

    return {
        "file":               os.path.basename(filepath),
        "spectral_flatness":  sf,
        "spectral_entropy":   se,
        "correlation_peak":   cp_,
        "power_variance":     pv,
        "burst_count":        bc,
        "burst_dur_mean_ms":  bd_mean,
        "burst_dur_std_ms":   bd_std,
        "duty_cycle":         dc,
        "micro_doppler_mean": md_mean,
        "micro_doppler_std":  md_std,
        "blade_rate_hz":      blade,
        "doppler_mean_hz":    dop_mean,
        "doppler_std_hz":     dop_std,
        "range_peak_bin":     r_bin,
        "range_peak_power":   r_pwr,
    }


In [6]:
# ============================================================
# Per-file worker
# ============================================================

def process_file(filepath):
    t_start = time.time()
    fname   = os.path.basename(filepath)
    try:
        feats    = extract_features(filepath)
        duration = time.time() - t_start
        return (fname, True, feats, duration)
    except Exception as e:
        import traceback
        return (fname, False,
                str(e) + "\n" + traceback.format_exc(),
                time.time() - t_start)

In [7]:
# ============================================================
# Batch processor
# ============================================================

def run_feature_extraction():

    all_files = []
    for root, dirs, files in os.walk(PROCESSED_DIR):
        for f in files:
            if f.endswith("_processed.npy"):
                all_files.append(os.path.join(root, f))

    if not all_files:
        print(f"No processed .npy files found in: {PROCESSED_DIR}")
        return

    print("=" * 65)
    print("  FEATURE EXTRACTION PIPELINE  —  GPU ACCELERATED")
    print("=" * 65)
    print(f"  Input dir   : {PROCESSED_DIR}")
    print(f"  Output CSV  : {OUTPUT_CSV}")
    print(f"  Files       : {len(all_files)}")
    print(f"  Max samples : {MAX_SAMPLES/1e6:.0f}M per file "
          f"(subsampled from 100M)")
    print(f"  Workers     : {MAX_WORKERS}")
    print("=" * 65)
    print()

    all_features = []
    success      = 0
    skipped      = 0
    t_total      = time.time()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(process_file, fp): fp
            for fp in all_files
        }
        for future in as_completed(futures):
            fname, ok, result, duration = future.result()
            if ok:
                all_features.append(result)
                print(f"  [✓] {fname}  ({duration:.1f}s)")
                success += 1
            else:
                print(f"  [✗] {fname}")
                print(f"      {result}")
                skipped += 1

    if all_features:
        df   = pd.DataFrame(all_features)
        cols = ["file"] + [c for c in df.columns if c != "file"]
        df   = df[cols]
        df.to_csv(OUTPUT_CSV, index=False)

        print()
        print("=" * 65)
        print(f"  COMPLETE")
        print(f"  Success : {success}  |  Failed : {skipped}")
        print(f"  Total   : {(time.time()-t_total)/60:.1f} min")
        print(f"  Shape   : {df.shape}")
        print(f"  Output  : {OUTPUT_CSV}")
        print()
        print("  Feature preview:")
        print(df.drop(columns=["file"]).describe().round(4).to_string())
        print("=" * 65)
    else:
        print("  No features extracted — check errors above.")


if __name__ == "__main__":
    run_feature_extraction()

  FEATURE EXTRACTION PIPELINE  —  GPU ACCELERATED
  Input dir   : C:\Users\RF-LAB\Desktop\Pipeline\Train_iq_preprocessed
  Output CSV  : C:\Users\RF-LAB\Desktop\Pipeline\Features_DroneRF.csv
  Files       : 454
  Max samples : 5M per file (subsampled from 100M)
  Workers     : 1

  [✓] 10100H_0_processed.npy  (3.5s)
  [✓] 10100H_10_processed.npy  (0.8s)
  [✓] 10100H_11_processed.npy  (1.0s)
  [✓] 10100H_12_processed.npy  (1.5s)
  [✓] 10100H_13_processed.npy  (1.2s)
  [✓] 10100H_14_processed.npy  (0.9s)
  [✓] 10100H_15_processed.npy  (0.8s)
  [✓] 10100H_16_processed.npy  (0.7s)
  [✓] 10100H_17_processed.npy  (1.4s)
  [✓] 10100H_18_processed.npy  (1.0s)
  [✓] 10100H_19_processed.npy  (0.7s)
  [✓] 10100H_1_processed.npy  (0.7s)
  [✓] 10100H_20_processed.npy  (0.7s)
  [✓] 10100H_2_processed.npy  (0.6s)
  [✓] 10100H_3_processed.npy  (0.7s)
  [✓] 10100H_4_processed.npy  (0.7s)
  [✓] 10100H_5_processed.npy  (0.7s)
  [✓] 10100H_6_processed.npy  (0.6s)
  [✓] 10100H_7_processed.npy  (0.7s)
  [✓]

In [8]:
import pandas as pd

df = pd.read_csv(r"C:\Users\RF-LAB\Desktop\Pipeline\Features_DroneRF.csv")

print(f"Total rows : {len(df)}")
print(f"NaN rows   : {df.isnull().all(axis=1).sum()}")
print(f"\nSample filenames:")
print(df['file'].dropna().head(20).tolist())

Total rows : 454
NaN rows   : 0

Sample filenames:
['10100H_0_processed.npy', '10100H_10_processed.npy', '10100H_11_processed.npy', '10100H_12_processed.npy', '10100H_13_processed.npy', '10100H_14_processed.npy', '10100H_15_processed.npy', '10100H_16_processed.npy', '10100H_17_processed.npy', '10100H_18_processed.npy', '10100H_19_processed.npy', '10100H_1_processed.npy', '10100H_20_processed.npy', '10100H_2_processed.npy', '10100H_3_processed.npy', '10100H_4_processed.npy', '10100H_5_processed.npy', '10100H_6_processed.npy', '10100H_7_processed.npy', '10100H_8_processed.npy']
